# Music Generation AI

In [1]:
!pip install music21 tensorflow keras numpy -q

In [2]:
import numpy as np
import os
import glob
import pickle

from music21 import converter
from music21 import instrument
from music21 import note
from music21 import chord
from music21 import stream

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import ModelCheckpoint

In [3]:
from google.colab import files
uploaded = files.upload()

import zipfile
import os

zip_path = "archive (3).zip"

extract_path = "/content/midi_dataset"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset Extracted")

Saving archive (3).zip to archive (3).zip
Dataset Extracted


In [4]:
midi_files = glob.glob("/content/midi_dataset/**/*.mid", recursive=True)

print("Total MIDI files:", len(midi_files))

Total MIDI files: 292


In [5]:
notes = []

for file in midi_files[:200]:   # limit for faster training

    print("Parsing:", file)

    try:
        midi = converter.parse(file)

        parts = instrument.partitionByInstrument(midi)

        if parts:
            notes_to_parse = parts.parts[0].recurse()
        else:
            notes_to_parse = midi.flat.notes

        for element in notes_to_parse:

            if isinstance(element, note.Note):
                notes.append(str(element.pitch))

            elif isinstance(element, chord.Chord):

                notes.append('.'.join(str(n) for n in element.normalOrder))

    except:
        continue

print("Total Notes Extracted:", len(notes))

Parsing: /content/midi_dataset/granados/gra_esp_3.mid


/usr/local/lib/python3.12/dist-packages/music21/midi/translate.py:922: TranslateWarning: Unable to determine instrument from <music21.midi.MidiEvent SEQUENCE_TRACK_NAME, track=4, data=b'Copyright \xa9 2001 by Bernd Kr\xfcger'>; getting generic Instrument
  warnings.warn(


Parsing: /content/midi_dataset/granados/gra_esp_2.mid
Parsing: /content/midi_dataset/granados/gra_esp_4.mid
Parsing: /content/midi_dataset/grieg/grieg_album.mid


/usr/local/lib/python3.12/dist-packages/music21/midi/translate.py:922: TranslateWarning: Unable to determine instrument from <music21.midi.MidiEvent SEQUENCE_TRACK_NAME, track=0, data=b'Grieg: Lyrische St\xfccke, Albumblatt, Opus 42 Nr. 2'>; getting generic Instrument
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/music21/midi/translate.py:922: TranslateWarning: Unable to determine instrument from <music21.midi.MidiEvent SEQUENCE_TRACK_NAME, track=5, data=b'Copyright \xa9 2010 by Bernd Krueger'>; getting generic Instrument
  warnings.warn(


Parsing: /content/midi_dataset/grieg/grieg_wanderer.mid


...

Parsing: /content/midi_dataset/schumann/scn16_8.mid
Parsing: /content/midi_dataset/schumann/schum_abegg.mid
Parsing: /content/midi_dataset/schumann/scn15_1.mid


/usr/local/lib/python3.12/dist-packages/music21/midi/translate.py:922: TranslateWarning: Unable to determine instrument from <music21.midi.MidiEvent SEQUENCE_TRACK_NAME, track=0, data=b'Von fremden L\xe4ndern und Menschen'>; getting generic Instrument
  warnings.warn(


Parsing: /content/midi_dataset/schumann/scn15_8.mid
Parsing: /content/midi_dataset/schumann/scn15_12.mid
Parsing: /content/midi_dataset/debussy/deb_prel.mid
Parsing: /content/midi_dataset/debussy/deb_menu.mid


/usr/local/lib/python3.12/dist-packages/music21/midi/translate.py:922: TranslateWarning: Unable to determine instrument from <music21.midi.MidiEvent SEQUENCE_TRACK_NAME, track=5, data=b'Copyright \xa9 1998 by Bernd Kr\xfcger'>; getting generic Instrument
  warnings.warn(


Total Notes Extracted: 4995


In [6]:
with open("notes.pkl", "wb") as filepath:
    pickle.dump(notes, filepath)

print("✅ Notes Saved")

✅ Notes Saved


In [7]:
sequence_length = 100

pitchnames = sorted(set(item for item in notes))

note_to_int = dict(
    (note, number) for number, note in enumerate(pitchnames)
)

network_input = []
network_output = []

for i in range(0, len(notes) - sequence_length):

    sequence_in = notes[i:i + sequence_length]

    sequence_out = notes[i + sequence_length]

    network_input.append(
        [note_to_int[char] for char in sequence_in]
    )

    network_output.append(
        note_to_int[sequence_out]
    )

n_patterns = len(network_input)

print("Total Patterns:", n_patterns)

# Reshape input
network_input = np.reshape(
    network_input,
    (n_patterns, sequence_length, 1)
)

# Normalize
network_input = network_input / float(len(pitchnames))

# One-hot encode output
network_output = to_categorical(network_output)

Total Patterns: 4895


In [8]:
model = Sequential()

model.add(
    LSTM(
        512,
        input_shape=(network_input.shape[1], network_input.shape[2]),
        return_sequences=True
    )
)

model.add(Dropout(0.3))

model.add(LSTM(512))

model.add(BatchNormalization())

model.add(Dropout(0.3))

model.add(Dense(256, activation='relu'))

model.add(Dense(len(pitchnames), activation='softmax'))

model.compile(
    loss='categorical_crossentropy',
    optimizer='adam'
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 100, 512)       │     1,052,672 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 100, 512)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 512)            │     2,099,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 185)            │        47,545 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,332,793 (12.71 MB)

 Trainable params: 3,331,769 (12.71 MB)

 Non-trainable params: 1,024 (4.00 KB)

In [11]:
history = model.fit(
    network_input,
    network_output,
    epochs=50,
    batch_size=64
)

Epoch 1/50
77/77 ━━━━━━━━━━━━━━━━━━━━ 4s 49ms/step - loss: 4.0666
Epoch 2/50
77/77 ━━━━━━━━━━━━━━━━━━━━ 4s 48ms/step - loss: 4.0173
Epoch 3/50
77/77 ━━━━━━━━━━━━━━━━━━━━ 4s 50ms/step - loss: 3.9510
Epoch 4/50
77/77 ━━━━━━━━━━━━━━━━━━━━ 4s 49ms/step - loss: 3.8675
Epoch 5/50
77/77 ━━━━━━━━━━━━━━━━━━━━ 4s 49ms/step - loss: 3.7918
Epoch 6/50
77/77 ━━━━━━━━━━━━━━━━━━━━ 4s 51ms/step - loss: 3.7265
Epoch 7/50
77/77 ━━━━━━━━━━━━━━━━━━━━ 4s 49ms/step - loss: 3.6377
Epoch 8/50
77/77 ━━━━━━━━━━━━━━━━━━━━ 4s 50ms/step - loss: 3.5997
Epoch 9/50
77/77 ━━━━━━━━━━━━━━━━━━━━ 4s 51ms/step - loss: 3.4988
Epoch 10/50
77/77 ━━━━━━━━━━━━━━━━━━━━ 4s 51ms/step - loss: 3.4135
Epoch 11/50
77/77 ━━━━━━━━━━━━━━━━━━━━ 4s 51ms/step - loss: 3.3280
Epoch 12/50
77/77 ━━━━━━━━━━━━━━━━━━━━ 4s 51ms/step - loss: 3.2428
Epoch 13/50
77/77 ━━━━━━━━━━━━━━━━━━━━ 4s 52ms/step - loss: 3.1410
Epoch 14/50
77/77 ━━━━━━━━━━━━━━━━━━━━ 4s 51ms/step - loss: 3.0608
Epoch 15/50
77/77 ━━━━━━━━━━━━━━━━━━━━ 4s 52ms/step - loss: 2.9831
Epoc

In [12]:
start = np.random.randint(0, len(network_input)-1)

int_to_note = dict(
    (number, note) for number, note in enumerate(pitchnames)
)

pattern = network_input[start]

prediction_output = []

for note_index in range(200):

    prediction_input = np.reshape(
        pattern,
        (1, len(pattern), 1)
    )

    prediction_input = prediction_input / float(len(pitchnames))

    prediction = model.predict(prediction_input, verbose=0)

    index = np.argmax(prediction)

    result = int_to_note[index]

    prediction_output.append(result)

    pattern = np.append(pattern, index)

    pattern = pattern[1:]

print("✅ Music Generated")

✅ Music Generated


In [13]:
offset = 0
output_notes = []

for pattern in prediction_output:

    # Chord
    if ('.' in pattern) or pattern.isdigit():

        notes_in_chord = pattern.split('.')

        notes_list = []

        for current_note in notes_in_chord:

            new_note = note.Note(int(current_note))

            new_note.storedInstrument = instrument.Piano()

            notes_list.append(new_note)

        new_chord = chord.Chord(notes_list)

        new_chord.offset = offset

        output_notes.append(new_chord)

    # Single note
    else:

        new_note = note.Note(pattern)

        new_note.offset = offset

        new_note.storedInstrument = instrument.Piano()

        output_notes.append(new_note)

    offset += 0.5

midi_stream = stream.Stream(output_notes)

midi_stream.write('midi', fp='generated_music.mid')

print("✅ MIDI File Saved")

✅ MIDI File Saved


In [14]:
from google.colab import files

files.download("generated_music.mid")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>